# Task C — step 3: train h_ψ (Doob h-function) on draw B, diagnose on draw C

Pre-registered in `DECISIONS.md` §11; results in §12. `h_ψ(z,τ) = h_min + (1−h_min)·exp(ᾱ_τ·s_ψ(z,τ))`, MSE on h against L* directly, full τ, uniform, h_min = 0.05, lr 1e-3→1e-5 cosine + EMA. Targets L* come from A's β* and ψ̂ (never renormalized). Everything reported is on draw C. Cell 0 is the Colab cell; locally skip it.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, json, pickle, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, torch
import matplotlib.pyplot as plt
from dataclasses import replace
import taskc
from taskc.config import CFG
from taskc.ptheta import load_checkpoint, load_draw, make_schedule
from config import q_params, CONSTRAINT_LEVELS, EXOTICS     # taskb
from constraints import build                              # taskb

RUN_TAG, LEVEL = "lrema", "C3"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
RUN = replace(CFG, artifact_dir=CFG.run_dir(RUN_TAG))
eps_model, std, _, _ = load_checkpoint(os.path.join(RUN.artifact_dir, RUN.ckpt_name), device=DEVICE)
sched = make_schedule(RUN, device=DEVICE)
tilt = pickle.load(open(os.path.join(RUN.artifact_dir, "dual", "tilts.pkl"), "rb"))[LEVEL]
q = q_params()
print("device:", DEVICE, "| P_theta:", RUN_TAG, "| level:", LEVEL)

In [ ]:
from taskc.hnet import HNet, train_hnet, tower_curve, h0_vs_L, grad_ratio_curve, save_hnet, load_hnet
from taskc.run_hpsi import targets_for
TRAIN  = True         # False -> load artifacts_taskc/<tag>/hpsi/hpsi_<LEVEL>.pt
EPOCHS = 100
H_MIN  = 0.05
out_dir = os.path.join(RUN.artifact_dir, "hpsi"); os.makedirs(out_dir, exist_ok=True)
B, C = load_draw(RUN, "B"), load_draw(RUN, "C")
LB, LC = targets_for(tilt, std, B.z, q), targets_for(tilt, std, C.z, q)
print(f"targets on B: E={LB.mean():.4f} min={LB.min():.4f} max={LB.max():.3f} | on C: E={LC.mean():.4f} min={LC.min():.4f} max={LC.max():.3f}")
print(f"h_min = {H_MIN} ({LB.min()/H_MIN:.1f}x below the smallest target on B)")

## Train (or load)

In [ ]:
if TRAIN:
    hnet = HNet(CFG.data_dim, 256, 32, H_MIN)
    log = train_hnet(hnet, B.z, LB, sched, device=DEVICE, epochs=EPOCHS)
    print(f"trained in {log.seconds/60:.1f} min; MSE {log.epoch_loss[0]:.4f} -> {log.epoch_loss[-1]:.4f}; floor active max {max(log.floor_frac):.1e}; clamp max {max(log.clamp_frac):.1e}")
    save_hnet(os.path.join(out_dir, f"hpsi_{LEVEL}.pt"), hnet, dict(level=LEVEL, epochs=EPOCHS, h_min=H_MIN, device=DEVICE,
              epoch_loss=log.epoch_loss, floor_frac=log.floor_frac, clamp_frac=log.clamp_frac, train_seconds=log.seconds))
    plt.figure(figsize=(5,3)); plt.plot(log.epoch_loss); plt.xlabel("epoch"); plt.ylabel("MSE(h, L*)"); plt.yscale("log"); plt.grid(alpha=.3); plt.show()
else:
    hnet = load_hnet(os.path.join(out_dir, f"hpsi_{LEVEL}.pt"), device=DEVICE); print(hnet.meta)

## Diagnostics on draw C: tower property, h₀ vs L*, gradient ratio

In [ ]:
ts = [0, 5, 10, 20, 40, 60, 80, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 980, 998]
tower = tower_curve(hnet, C.z, sched, ts, device=DEVICE)
h0 = h0_vs_L(hnet, C.z, LC, sched, device=DEVICE)
gr = grad_ratio_curve(hnet, eps_model, C.z, sched, ts, device=DEVICE)
print(f"tower: max |E_C[h_tau]-1| = {max(abs(tower[t]['mean']-1) for t in ts):.4f}  (pre-registered band: 0.01 for tau>300, 0.03 near 0)")
print(f"h_0 vs L*: corr {h0['corr']:.4f}  R2 {h0['r2']:.4f}  RMSE {h0['rmse']:.4f}")
print(f"grad ratio: peak {max(gr[t]['ratio_mean'] for t in ts):.4f} at tau={max(ts, key=lambda t: gr[t]['ratio_mean'])};  at tau=900: {gr[900]['ratio_mean']:.4f}")
print(f"eps-space correction |sqrt(1-abar) grad log h|: peak {max(gr[t]['eps_corr_norm'] for t in ts):.4f} at tau={max(ts, key=lambda t: gr[t]['eps_corr_norm'])}")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].errorbar(ts, [tower[t]['mean'] for t in ts], yerr=[2*tower[t]['se'] for t in ts], marker="o", ms=3); ax[0].axhline(1, c="k", lw=.8); ax[0].axhspan(0.99, 1.01, alpha=.1); ax[0].set_title("tower: E_C[h_tau(Z_tau)]"); ax[0].set_xlabel("tau")
ax[1].plot(ts, [gr[t]['ratio_mean'] for t in ts], marker="o", ms=3, label="||grad log h|| / ||grad log p||"); ax[1].plot(ts, [gr[t]['eps_corr_norm'] for t in ts], marker="x", ms=3, label="||eps correction||"); ax[1].set_xlabel("tau"); ax[1].legend(); ax[1].set_title("correction size vs tau")
with torch.no_grad():
    zC = torch.from_numpy(C.z[:20000]).to(DEVICE); hh = hnet(zC, torch.full((20000,), 0.5/sched.T, device=DEVICE), sched.alphas_bar[0].expand(20000)).cpu().numpy()
ax[2].scatter(LC[:20000], hh, s=2, alpha=.3); ax[2].plot([0, LC.max()], [0, LC.max()], "k--", lw=1); ax[2].set_xlabel("L*(x)"); ax[2].set_ylabel("h_0(z_0)"); ax[2].set_title(f"h_0 vs L* (corr {h0['corr']:.3f})"); ax[2].set_xscale("log"); ax[2].set_yscale("log")
plt.tight_layout(); plt.show()

Stop. Results go to `DECISIONS.md` §12; next is `taskc_05_smt.ipynb` (corrected sampler + amortization test).